# Run underfit in Colab

**by [@dadabots](https://dadabots.com)** · last updated: 2026-05-26 13:39 UTC
[github.com/dada-bots/underfit](https://github.com/dada-bots/underfit)

LoRA finetuning for Stable Audio 3, in a Colab notebook.

> ⚠️ **Before running any cells:** set the runtime to GPU.
> **Runtime → Change runtime type →** pick a GPU.
>
> **Recommended: H100** — underfit runs *much* better on H100. Training is fast, demos finish in seconds, the dashboard stays responsive. It costs $9.99 of Colab compute credits — see [colab.research.google.com/signup](https://colab.research.google.com/signup). L4 / A100 / G4 are also fine.
>
> **Colab Free (T4) is supported but slow.** Training works, but each step takes much longer and you may see warnings. It requires patience, but it's possible. On T4, recommended to use minimal settings: **SA3-small** (medium can still work, but it's slow), a **short latent sequence length** (e.g. 128), and **2 demos per checkpoint** (the Two preset).

>
> **Strongly recommended: use ngrok in Step 4.** The dashboard UI can feel laggy and occasionally freezes on Colab because Colab's built-in port-proxy buffers HTTP responses (audio playback is the worst: the proxy holds the whole audio file before forwarding instead of streaming it). *Training itself is unaffected* — it runs as a detached subprocess on the GPU and survives dashboard freezes or closed tabs. To smooth out the UI, sign up free at [ngrok.com](https://ngrok.com) and paste your auth token into the `NGROK_AUTHTOKEN` field in Step 4. If anything ever freezes, re-run Step 4 to restart the server. For the smoothest experience overall, running underfit on a normal Linux GPU box is *dramatically* more stable than Colab — see the README on GitHub.

The dashboard runs *inside* the Colab VM on `localhost:8787` and is exposed via the Colab port-proxy (private to your Google account) or via ngrok (public link).

Run each cell with `Shift+Enter`.


## 1. Install underfit

In [ ]:
#@title Step 1 — Install underfit (clone + sync) { display-mode: "form" }
# ── Pre-flight: confirm GPU runtime is attached ─────────────────────────
# Uses shutil.which() to test for `nvidia-smi` BEFORE calling it, so the
# 'No such file or directory' FileNotFoundError doesn't blow up with
# IPython's traceback formatter (Python 3.12 + IPython interact badly on
# unhandled exits). Everything below the check is gated behind a plain
# if/else — no `raise SystemExit()` — so a missing GPU is a clean message,
# not a stack trace.
import os, shutil, subprocess

gpu_name = None
if shutil.which("nvidia-smi"):
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name",
                            "--format=csv,noheader"],
                           capture_output=True, text=True, timeout=5)
        if r.returncode == 0 and r.stdout.strip():
            gpu_name = r.stdout.strip().splitlines()[0]
    except Exception:
        pass

if gpu_name is None:
    print()
    print("═" * 70)
    print("  ⚠️  NO GPU DETECTED — underfit needs a GPU runtime  ⚠️")
    print("═" * 70)
    print()
    print("  Fix it now:")
    print("    1. Runtime → Change runtime type")
    print("    2. Hardware accelerator → T4 GPU  (or L4 / H100 if you have Pro)")
    print("    3. Click Save (your VM will reconnect)")
    print("    4. Re-run this cell")
    print()
    print("═" * 70)
else:
    print(f"✓ GPU detected: {gpu_name}\n")
    # ── Clone underfit + install deps (uv + venv, ~5 GB of wheels, ~1 min) ──
    if not os.path.isdir("/content/underfit"):
        subprocess.run(["git", "clone",
                        "https://github.com/dada-bots/underfit",
                        "/content/underfit"], check=True)
    os.chdir("/content/underfit")
    subprocess.run(["./install.sh", "--no-setup"], check=True)


## 2. Mount Drive + HF auth + state dir

Splits storage between Drive (persistent, slower) and local SSD (ephemeral, fast).

<details>
<summary><b>How storage is split</b> — click to expand</summary>

| Lives on | Why |
|---|---|
| **Drive** (`UNDERFIT_STATE_DIR`) | runs.json, datasets, audio, gradio logs — small files, must persist across session resets |
| **Local SSD** (`UNDERFIT_MODELS_DIR=/content/models`) | Model files used by training. Reads at ~500 MB/s vs Drive's ~30 MB/s. Wiped on session reset, but re-fetched from HF in ~2–3 min. |
| **Local SSD** (`HF_HOME=/content/hf-cache`) | HuggingFace download cache. Stays on local SSD because re-downloading from HF (~150 s) is faster than copying from a Drive-cached copy (~500 s). |

</details>


In [ ]:
#@title Step 2 — Mount Drive, configure storage, HuggingFace auth { display-mode: "form" }
import os
from google.colab import drive
drive.mount('/content/drive')

# Persistent user state (runs, datasets, audio) — lives on Drive so it
# survives Colab session resets.
os.environ['UNDERFIT_STATE_DIR'] = '/content/drive/MyDrive/underfit-state'

# Model files — stay on local SSD for fast reads (~500 MB/s vs Drive's ~30 MB/s
# through FUSE). Wiped on session reset, but re-fetched in ~2–3 min by the wizard.
os.environ['UNDERFIT_MODELS_DIR'] = '/content/models'

# Training logs — write to local SSD for instant dashboard reads. A background
# thread launched in Step 4 rsyncs to Drive every ~60s for durability.
os.environ['UNDERFIT_LOGS_DIR'] = '/content/underfit-logs'

# State JSON files (runs.json, datasets.json, gradio_*.json) — also on local
# SSD. Synced to Drive every ~10s. Dashboard uses atomic tmp+rename writes so
# the sync thread never reads a torn file.
os.environ['UNDERFIT_STATE_FILES_DIR'] = '/content/underfit-state'

# Seed-LoRA upload staging — local SSD so uploads land in seconds instead of
# minutes. The file gets copied into the run dir (on Drive) at launch time
# anyway, so this dir doesn't need durability.
os.environ['UNDERFIT_SEED_LORAS_DIR'] = '/content/seed_loras'

# HuggingFace cache — also on local SSD. Counter-intuitive: caching on Drive seems
# like it would save downloads, but Drive write is ~5–15 MB/s, so a cache-then-
# copy round-trip is ~3× slower than a fresh HF download each session.
os.environ['HF_HOME'] = '/content/hf-cache'

# HuggingFace auth — needs access to gated stabilityai/stable-audio-3-* repos.
# Token lives on Drive (/content/drive/MyDrive/hf-cache/stored_tokens) so it
# persists across Colab session resets. On each session we copy it to local
# SSD's HF_HOME, validate, and only prompt for a fresh login if the existing
# token is missing or invalid.
import shutil
from pathlib import Path
from huggingface_hub import whoami

_drive_hf_dir = Path('/content/drive/MyDrive/hf-cache')
_local_hf_dir = Path(os.environ['HF_HOME'])
_local_hf_dir.mkdir(parents=True, exist_ok=True)

# Seed local from Drive (both file names — newer huggingface_hub uses
# stored_tokens; older uses plain 'token')
for fname in ('stored_tokens', 'token'):
    src_f = _drive_hf_dir / fname
    dst_f = _local_hf_dir / fname
    if src_f.exists() and not dst_f.exists():
        shutil.copy2(src_f, dst_f)

# Validate
_user = None
try:
    _user = whoami().get('name')
except Exception:
    _user = None

if _user:
    print(f'✓ Logged in as {_user}  (token loaded from Drive)')
else:
    print('No valid HuggingFace token found. Get one at https://huggingface.co/settings/tokens')
    print('When asked "Add token as git credential?", you can say N.')
    !hf auth login
    # Persist the freshly-saved token to Drive for future sessions
    _drive_hf_dir.mkdir(parents=True, exist_ok=True)
    for fname in ('stored_tokens', 'token'):
        src_f = _local_hf_dir / fname
        dst_f = _drive_hf_dir / fname
        if src_f.exists():
            shutil.copy2(src_f, dst_f)
    print(f'✓ Token saved to {_drive_hf_dir} — future sessions will skip the login prompt.')


## 3. Pick a backend + download model packs

This clones the Stable Audio 3 backend into `/content/stable-audio-3`, installs it into the venv, and downloads the model packs you check below.

> ⚠️ **First time only:** open the link below and click **"Agree and access repository"**. The three ARC repos share one license — agreeing to any one unlocks all three. The base repos aren't gated. Without this, the download in this cell will fail with `401 Unauthorized`. (Approval is instant once you accept the license.)
>
> - [Agree to the SA3 license here](https://huggingface.co/stabilityai/stable-audio-3-medium)

In [ ]:
#@title Step 3 — Download SA3 backend + model packs { display-mode: "form" }
#@markdown Pick which model pack(s) to download (~6–9 GB each — only pick what you need):
download_sa3_medium = True  #@param {type:"boolean"}
download_sa3_sm_music = False  #@param {type:"boolean"}
download_sa3_sm_sfx = False  #@param {type:"boolean"}

# Clone the SA3 backend if not already present
import os
if not os.path.isdir("/content/stable-audio-3"):
    !git clone https://github.com/Stability-AI/stable-audio-3.git /content/stable-audio-3

# Build the comma-separated --models list from the form toggles
_picks = []
if download_sa3_medium:   _picks.append("sa3-medium")
if download_sa3_sm_music: _picks.append("sa3-sm-music")
if download_sa3_sm_sfx:   _picks.append("sa3-sm-sfx")
if not _picks:
    raise SystemExit("Pick at least one model pack above and re-run this cell.")
_models_arg = ",".join(_picks)
print(f"Downloading: {_models_arg}")

# Run the setup wizard non-interactively
!uv run python -m underfit.cli.setup \
    --backend sa3 \
    --backend-path /content/stable-audio-3 \
    --models {_models_arg}


## 4. Launch the dashboard

Two ways to reach the dashboard once it's running:

- **Private (default):** Colab's built-in port-proxy. Only your Google account can open the URL. No third-party service.
- **Shareable:** ngrok tunnel — public URL anyone with the link can use. Free tier requires a token from [ngrok.com](https://ngrok.com/signup).

This cell **completes** after the dashboard starts (the subprocess keeps running in the background). Step 5 is a separate cell that polls the dashboard for status updates.


In [ ]:
#@title Step 4 — Launch the dashboard { display-mode: "form" }
#@markdown **Strongly recommended:** check `USE_NGROK` and paste your
#@markdown auth token. Colab's built-in port-proxy buffers HTTP
#@markdown responses — the dashboard feels laggy and *fully freezes*
#@markdown during audio playback (the proxy buffers the entire audio
#@markdown file before forwarding to your browser, blocking every
#@markdown other request until done). Free ngrok signup gives you a
#@markdown public URL that streams properly. Token lives at
#@markdown [dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken).
import subprocess
from underfit.monitor import launch_dashboard_subprocess, dashboard_button, restart_dashboard_button, start_drive_log_sync, start_drive_state_sync

USE_NGROK = True #@param {type:"boolean"}
NGROK_AUTHTOKEN = "" #@param {type:"string"}

# Sync EVERYTHING from Drive → local SSD BEFORE launching the dashboard.
# Two reasons to do both seeds up-front:
#   1. The dashboard's RunsRegistry/_load reads runs.json + tails the
#      latest log of every active run on startup. If logs aren't seeded
#      yet, those reads either come up empty or fall back to Drive
#      (slow) — visible as a long first-paint and a long first /api/stream.
#   2. After both seeds, we know the dashboard's working set is hot on SSD.
# Both functions do a synchronous cold-start seed before they return; the
# background sync threads they spawn keep things in sync afterwards.
start_drive_state_sync(
    local_dir='/content/underfit-state',
    drive_dir='/content/drive/MyDrive/underfit-state',
    interval=10,
)
start_drive_log_sync(
    local_dir='/content/underfit-logs',
    drive_dir='/content/drive/MyDrive/underfit-state/runs',
    interval=60,
)

# launch_dashboard_subprocess() kills any prior process bound to :8787,
# launches dashboard/server.py with start_new_session=True (so cell
# interrupts don't propagate), and blocks until the "Dashboard running on"
# ready-marker. Returns the Popen handle.
proc = launch_dashboard_subprocess(port=8787)

# Render the big "Open Dashboard" button.
# - With ngrok (recommended): public URL that streams responses properly.
# - Without ngrok: falls back to Colab's port-proxy, which buffers and
#   will cause freezes during audio playback. Works, but worse.
if USE_NGROK and NGROK_AUTHTOKEN:
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyngrok"])
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    ngrok_url = ngrok.connect(8787, 'http').public_url
    dashboard_button(url=ngrok_url, label="🌐 Open Dashboard (via ngrok)")
elif USE_NGROK and not NGROK_AUTHTOKEN:
    print("⚠ USE_NGROK is checked but NGROK_AUTHTOKEN is empty.")
    print("  Grab a free token at https://dashboard.ngrok.com/get-started/your-authtoken,")
    print("  paste it into the form field above, and re-run this cell.")
    print("  Falling back to Colab's port-proxy for now (expect freezes during audio playback).")
    dashboard_button(port=8787)
else:
    print("ℹ Using Colab's built-in port-proxy. The dashboard may freeze")
    print("  during audio playback because Colab buffers entire response")
    print("  bodies before forwarding. Enable USE_NGROK above for a")
    print("  dramatically smoother experience.")
    dashboard_button(port=8787)

# Secondary action below the big Open button.
restart_dashboard_button(port=8787)

print("\n(Dashboard is running in the background. Move on to Step 5 to monitor it.)")


## 5. Monitor the dashboard (optional)

Polls the dashboard's API every 10 seconds and shows: runs by status, dataset count, active gradio instances. Doubles as a keep-alive — it keeps the cell running so Colab sees the session as active.

If the dashboard subprocess crashes (or gets OOM-killed mid-training), the monitor **auto-restarts it** after ~30 s of consecutive failed polls. You won't lose your runs — `runs.json` lives on Drive, the dashboard re-reads it on launch.

Stop the monitor anytime with ⏹ / `Ctrl+M I` — the dashboard keeps running. Re-run this cell to restart monitoring.

In [ ]:
#@title Step 5 — Monitor the dashboard (optional, stoppable) { display-mode: "form" }
from underfit.monitor import launch_dashboard_subprocess, start_monitor

def _restart_dashboard():
    """Called by the monitor after ~30s of failed polls. Only restarts if the
    subprocess is actually dead — otherwise the dashboard is alive but slow
    under load (training hammering GPU/CPU), and killing it makes things worse."""
    global proc
    if proc.poll() is None:
        print("ℹ  Dashboard slow but PID is still alive — skipping restart.", flush=True)
        return
    print(f"⚠️  Dashboard subprocess died (returncode={proc.returncode}). Restarting…", flush=True)
    proc = launch_dashboard_subprocess(port=8787, quiet=True)
    print("✓ Dashboard restarted (same URL — Step 4 button still works).\n", flush=True)

# Blocking: this cell stays "running" in Colab while polling every 10s.
# Keeps the comm channel active AND keeps Colab's session UI showing
# the cell as busy, both of which help against the idle timeout.
# Stop with ⏹ / Ctrl+M I; the dashboard keeps running. To restart the
# dashboard server itself, re-run Step 4 (training runs aren't affected).
start_monitor(interval=10, on_unreachable=_restart_dashboard)


## 6. Create your first dataset

In the dashboard tab, click **"+ Dataset"** to encode a folder of audio into pre-encoded latents.

### Get your audio onto Google Drive

1. Open [drive.google.com](https://drive.google.com) in another tab.
2. Drag-and-drop a folder of audio (e.g. `my-songs/`) into "My Drive".
3. Wait for the green check on every file.
4. In the dashboard's **Create Dataset** form, paste the path:
   ```
   /content/drive/MyDrive/my-songs
   ```
   (replace `my-songs` with your folder's actual name)

Verify the path from a Colab cell if you want:
```python
!ls /content/drive/MyDrive/my-songs | head
```

### Tips on what to feed it

- **10+ minutes** of audio is the floor; **30+ min** is better. Quality > quantity.
- **WAV / FLAC / MP3 / OGG / OPUS / M4A / AIFF** all work.
- **One coherent style** per dataset (one artist, one genre, one SFX category).
- The dashboard lets you tick/untick individual files after the scan — you don't have to pre-curate.

### Adding metadata for prompts (optional)

Each clip can have key-value metadata that the LoRA learns to associate with the audio. Underfit looks in this order:

**1. JSON sidecar** (winner — overrides any embedded tags). A file with the same name as the audio, `.json` extension, sitting either:

  - **right next to the audio**: `my-songs/01.wav` + `my-songs/01.json`, or
  - **in a sibling `json/` folder**: `my-songs/01.wav` + `my-songs/json/01.json`

  Any string/number values are kept — both standard keys and custom ones. Useful for per-clip captions, BPMs scraped from somewhere, prompt overrides, anything:

  ```json
  {
    "title": "Eclipse",
    "genre": "deathcore",
    "bpm": 145,
    "mood": "melancholic",
    "prompt": "slow blackened deathcore with clean choir vocals"
  }
  ```

**2. Embedded audio tags** (ID3 on MP3, Vorbis on FLAC/OGG, M4A atoms, etc.) — read via the `audio_metadata` library. Underfit extracts these keys if present: `title`, `artist`, `album`, `genre`, `label`, `date`, `composer`, `bpm`. So if your music library is already tagged in Picard / Mp3tag / iTunes, you get those for free.

**3. Skip metadata entirely** — totally fine. In **New Finetune**'s *Configure prompts* step you can compose prompts from:

  - the relative file path (e.g. `/sfx/explosions/fireworks/03.wav` becomes the prompt — directory structure becomes meaning), or
  - a fixed string for every clip (e.g. one trigger phrase), or
  - both, mixed.

  For a single-artist or single-genre dataset this is often all you need. For SFX organized by category, path-as-prompt with folder names already encoding the meaning is the easy default.

### What happens when you submit

The dashboard spawns a pre-encoding subprocess on the GPU. Each audio file becomes a `.npy` (encoded latent) + `.json` (metadata) pair. When the dataset shows up in the "Datasets" panel, it's ready to train against.

> ⏳ **On free-tier Colab T4 this can take a while.** Just leave it running until the dashboard says **Done!** — the encoding panel updates as each file finishes.


## 7. Train your first LoRA

In the dashboard tab, click **"+ Finetune"** (or "New finetune"). This kicks off a training subprocess that watches your dataset and produces a small adapter `.safetensors` file you can use in inference.

### Fill in the form

| Field | What to put | Why |
|---|---|---|
| **Name** | `my-first-lora` (or whatever — alphanumeric + hyphens) | Used as the run ID and the `.safetensors` filename |
| **Model** | `sa3-medium` (the one you downloaded in Step 3) | Base model to finetune against |
| **Dataset** | the one you created in Step 6 | Pre-encoded latents to train on |
| **LoRA type** | **`DoRA`** (recommended) | Generally better-quality fits than plain LoRA. `DoRA-XS` is the small-file variant. |
| **LoRA rank** | `16` | Capacity knob. 16 is a sane default. Higher = more parameters (heavier file, slower train, more overfitting risk). |
| **Steps** | `10000` | A reasonable LoRA lands around **10k steps** — that's where it *creatively underfits*: still general enough to vary on new prompts, not yet memorizing the training set. Past 10–20k it overfits and starts regurgitating. |
| **Batch size** | `1` on T4, up to `8` on H100 | Bigger = better gradients but more VRAM. T4 caps at 1 for SA3-medium fp16. |
| **Latent length** | model default, or shorter | You can train a LoRA at a shorter duration than the base model's max. Training is faster, the LoRA overfits less, and *very short* sequence LoRAs are especially useful for audio2audio style-transfer. Lower this to e.g. `128` for quicker iteration. |
| **Learning rate** | leave default | Usually fine; lower if you see grad-norm spikes |
| **Demo every** | `500` (or `1000`) | How often to generate audio previews during training. More frequent = more disk + more wall-clock spent on demos. |
| **Checkpoint every** | `500` | How often to save a `.safetensors` file. Each one is restartable. |

**Tips:**

- **Latent length is the underrated knob.** Lowering it to ~47 s or ~12 s (with random_crop on) is often the cleanest way to learn a style without memorization. The model only ever sees patterns at that timescale and never sees the full structure of your songs, so it can't memorize structure. Listen to the demos at full length to hear how it extrapolates.
- **Batch size is a creative parameter.** Just because you have VRAM for 8 doesn't mean you want 8. `batch_size=1` learns something different (focuses on one song at a time, sharper imprint) from `batch_size=4` (averages gradients across songs, smoother fit). Experiment.

Click **Next →** to walk through three more panels:

### Pick a GPU

Each GPU shows its current VRAM use plus an *estimate* of how much your run will need (based on model + LoRA rank + batch size). Pick one with headroom. If the estimate goes red, lower batch size or rank.

On Colab Free you'll only see one card. Don't worry about CUDA_VISIBLE_DEVICES — the wizard pins the chosen GPU automatically.

### Configure prompts

How prompts are built from your dataset's filenames and tags each training step. Three sources, with balance percentages that should sum to 100:

- **Tags** — metadata fields (album, year, genre, bpm) extracted from your audio files. Toggle which keys to include. Useful when your dataset has rich tags.
- **Paths** — directory + filename. Useful when your folder structure encodes meaning (e.g. `metal/death/archspire/`). Options to hide extensions, top-most directory, etc.
- **Fixed** — a literal string applied to every file. Use for single-artist or single-genre datasets where every example should share the same prompt.

Other knobs:
- **Trigger word** — optionally prepend a magic phrase to N% of prompts. The LoRA learns to associate the phrase with your style; at inference time, including the trigger word turns the style on. A short unique token like `mfd3` + `trigger_pct=80` works well.
- **Shuffle** — randomize the order of comma-separated parts each step (keeps the model from memorizing the comma order).

**Simplest config for a small single-style dataset (most LoRAs):** Fixed text = your trigger phrase, balance 100% fixed.

**A good mix to try:** ~50/50 fixed + tags (or fixed + paths) — training sees both the shared style anchor (your trigger phrase) and a per-song detail (title, BPM, genre, etc.). The LoRA learns the style as the constant signal *and* learns that the style coexists with varied per-song descriptors. At inference time you can either use just the trigger phrase (style only) or trigger + per-song details (style on a specific kind of song).

**You're blending two prompt vocabularies.** SA3's base model was trained with a specific prompt style — labelled key-value pairs like `Genre: techno, BPM: 140, Mood: dark`. Your LoRA prompts get composed *on top* of that vocabulary at inference time. Re-using the base model's prompt format seems likely to help the LoRA stack with what the base already knows, but you can also combine old and new prompt formats together (e.g. `/metal/song.mp3, BPM: 140`). Skim a few of your demo prompts in Step 4 to spot-check the format.

### Set up demos

The demos that get generated for you to listen to during training. You can:

- **Edit individual demos**: prompt text, sampler (RF or ARC), CFG, step count, seed. Each one runs once per `demo_every` cycle.
- **Re-generate prompts** with `↻` (per-demo) or the big `↻` button (all) to re-sample prompts from your dataset.
- **Add / remove** demos with the green `+` / red `−`.

### Click **Launch**.

### What you'll see during training

1. **Run appears** in the run panel with status `loading...` (5–15 s while the base model loads into VRAM)
2. Status flips to `training` and a **loss curve** starts plotting
3. Every `demo_every` steps, the run pauses for ~30 s to generate **4 demo MP3s** with their **spectrogram previews** (your monitor cell in Step 5 will tick `1 training` → keep an eye on it)
4. Every `checkpoint_every` steps, a fresh `.safetensors` lands and shows up in the checkpoints list

The dashboard tab is the live view. The status monitor cell in this notebook is the at-a-glance summary.

### How to know when to stop

Two signals — use both:

**1. Loss curve.** Watch the loss panel in the dashboard. The **elbow** — where it stops dropping steeply and starts to flatten — tends to be a good checkpoint to keep. Past the elbow you're refining, but also creeping toward memorization.

**2. Your ears, on the demos.** Open a demo MP3 every couple of thousand steps. What you're looking for in a healthy run:

- **Base RF demos (CFG≈7) light up first.** Around the time the run is starting to "get it," your CFG=7 demos suddenly sound right — clearly your training style on a coherent prompt.
- **Then CFG=7 over-cooks, and CFG=1 takes over.** Past the elbow, CFG=7 starts sounding artifacted / over-saturated. The lower-CFG demos (CFG≈1) keep improving and end up sounding the cleanest. If CFG=1 is sounding good and CFG=7 isn't, that's a sign the LoRA has internalized the style and no longer needs the prompt-classifier guidance to bring it out.
- **Conditional → unconditional crossover.** Early on, only the prompted demos sound like the training style. Later, even the **empty-prompt / unconditional** demos start sounding like the style — the model has "absorbed" the dataset, not just learned to recall it on cue.
- **ARC demos lag a bit but end up cleaner.** The ARC-distilled demos take a few thousand more steps to catch up to the base RF demos, but their final quality is usually better. Don't stop based on ARC alone too early; trust the base RF signal for "is this learning yet."

**3. Don't fear a memorized checkpoint — it might be exactly what you want.** "Overfitting" is only a problem if you're chasing creative variation. A memorized checkpoint can be incredibly useful:

- Dial **LoRA strength down** at inference time (in the gradio panel) — a memorized adapter at strength ~0.7 often gives the cleanest "in the style of" generations.
- **audio2audio / style-transfer** effects often hit harder with a memorized model — the strong style signal pulls the input audio into the training distribution more decisively.

So even past the elbow, save checkpoints. Different downstream uses want different points on the underfit ↔ memorize curve.

**Failure modes:**
- Demos sound **identical to the input** by 5k or so → overfitting too fast. Lower the rank or stop earlier.
- Demos sound **nothing like the input** even at 10k+ → dataset is too varied, too small, or LR is too low.

You can **stop the run anytime** (button in the dashboard) — the last checkpoint is yours. You can also **resume** from any checkpoint to train further.

### Launch a Gradio from any checkpoint

Every checkpoint in the dashboard has a 🚀 button. Click it to spin up a private Gradio inference UI for that exact `.safetensors`. The Gradio panel lets you:

- **Dial LoRA strength below 1.0** — blends the LoRA with the base model. A heavily-trained / "memorized" checkpoint often sounds best around 0.6–0.8, giving you "in the style of" without straight regurgitation.
- **LoRA interval (skip first step).** Skipping the LoRA on the very first denoising step lets the base model establish song structure before your style takes over — that first step determines the broad layout. Counterintuitively this often prevents the LoRA feeling muddy or static and reduces underfitting. Or use audio2audio to establish a structural template more directly.
- **Mix prompt vocabularies.** Combine SA3's base prompt format (`Genre: techno, BPM: 140, Mood: dark`) with whatever your LoRA learned (your trigger word, filename / path fragments, etc.). Both work; blends usually work best.
- **audio2audio.** Drop in a song; SA3 + your LoRA stylizes it.
- **Inpainting.** Paint over a region of an input and regenerate just that region in the LoRA's style.

Most knobs work the same as any SA3 inference UI; the LoRA-specific ones (strength, interval) are what give you fine-grained control over how much of the LoRA bleeds into the output.

### Get your LoRA out of Colab

1. In the dashboard's run panel, find the checkpoint you want
2. Click the download (⬇) button → grabs the `.safetensors` file (~20 MB for DoRA/LoRA at rank 16; DoRA-XS is smaller)
3. **Or** copy it from Drive:
   ```
   /content/drive/MyDrive/underfit-state/runs/<run-id>/<step>.safetensors
   ```

Now drop that `.safetensors` into any Stable Audio 3 inference setup (the dashboard's gradio panel, `python -m stable_audio_3`, ComfyUI's SA3 nodes, etc.) and the adapter layer-grafts onto SA3-medium at runtime.

**Beyond the dashboard's gradio panel, what you can do with a `.safetensors`:**

- **Blend multiple LoRAs at once.** `run_gradio.py` accepts multiple `--lora-ckpt-path` arguments. Train two LoRAs on different styles, load both at inference time, dial each strength <1.0, and get a fusion. Strength interacts non-linearly so it's worth experimenting.
- **Continue training from an existing LoRA.** In **New Finetune**, the "Start from a previous LoRA" upload picks up your `.safetensors` and keeps going. There's an art to mixing datasets this way — for example I trained a music-style DoRA (Dadabots baroquecore) for 10k steps at 47 s, then continued it on a production-quality DoRA (Encanti bass music) at 12 s for 100 steps. The result was one LoRA that fused both styles in a way neither alone could have produced.
- **Any SA3 inference setup can load it.** Plugins, ComfyUI nodes, custom Python wrappers — the file is the standard SA3 LoRA layout, so anything that supports SA3 LoRAs can use yours.


### What to try next

- **DoRA-XS at rank 32** — tiny files, more capacity than rank-16 DoRA.
- **More steps** (15k, 20k) — push closer to memorization to see what creative underfitting looks like vs. overfitting.
- **Different prompts in demos** — edit the demo prompt list to test generalization.
- **A larger / more varied dataset** — usually the highest-leverage knob.


## 8. Maintenance & debugging

Two utilities below — run them anytime, in any order.

- **Pull latest underfit code** — fetches the newest commit from GitHub. After running, click **🔄 Restart Dashboard** (Step 4) so the new server picks it up.
- **Diagnostic snapshot** — prints git HEAD, torch+CUDA, every GPU's compute capability, dashboard process, disk usage, the latest run's log + every sidecar, and a sanity import check. Paste the full output when reporting a problem.


In [ ]:
# Pull latest underfit code from GitHub.
# After this runs, click 🔄 Restart Dashboard (Step 4) so the new server.py is live.
%cd /content/underfit
!git pull
!git log -1 --oneline


In [ ]:
# One-shot diagnostic snapshot for bug reports.
from underfit.monitor import debug_info
debug_info()
